# `klygo.files` (`filesystem.py`): Filesystem inspection, traversal, and changes

Inspect directory entries, traverse trees, and make deliberate changes to files and folders.

Run this notebook from top to bottom. It creates only temporary example files.

In [ ]:
!pip install git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample files

Run this setup again to reset the examples. No Drive mount or manual upload is required.

In [1]:
from tempfile import TemporaryDirectory
from klygo import files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-files-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr


In [2]:
(root / "images").mkdir()
(root / "images" / "sample.jpg").write_bytes(b"image bytes")
(root / "images" / "second.jpg").write_bytes(b"other image bytes")
(root / "cache").mkdir()
(root / "cache" / "ignored.jpg").write_bytes(b"cached image")
print("Sample tree ready:", root)

Sample tree ready: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr


## Jump to an API

[`exists`](#exists) · [`is_file`](#is_file) · [`is_dir`](#is_dir) · [`is_symlink`](#is_symlink) · [`list_entries`](#list_entries) · [`find`](#find) · [`iter_find`](#iter_find) · [`walk`](#walk) · [`mkdir`](#mkdir) · [`copy`](#copy) · [`move`](#move) · [`rename`](#rename) · [`remove`](#remove)

<a id="exists"></a>

### `files.exists`

Check whether a filesystem path exists.

**Signature:** `files.exists(path: 'PathInput') -> 'bool'`

- `path`: Path to inspect or operate on.

**Returns:** `bool` — Whether the path exists.

A broken symlink reports `False` because the target does not exist; `is_symlink()` can still report `True` for its directory entry.

In [3]:
print(files.exists(root / 'images' / 'sample.jpg'))
print(files.exists(root / 'missing.jpg'))

True
False


**Another useful pattern**

In [4]:
planned = root / 'output' / 'result.json'
print('Before creation:', files.exists(planned))
files.mkdir(planned.parent)
planned.write_text('{}', encoding='utf-8')
print('After creation:', files.exists(planned))

Before creation: False
After creation: True


<a id="is_file"></a>

### `files.is_file`

Check whether a path identifies a regular file.

**Signature:** `files.is_file(path: 'PathInput') -> 'bool'`

- `path`: Path to inspect or operate on.

**Returns:** `bool` — Whether the path is a regular file.

In [5]:
print(files.is_file(root / 'images' / 'sample.jpg'))
print(files.is_file(root / 'images'))

True
False


**Another useful pattern**

In [6]:
for item in (root / 'images' / 'sample.jpg', root / 'images', root / 'missing'):
    print(item.name, files.is_file(item))

sample.jpg True
images False
missing False


<a id="is_dir"></a>

### `files.is_dir`

Check whether a path identifies a directory.

**Signature:** `files.is_dir(path: 'PathInput') -> 'bool'`

- `path`: Path to inspect or operate on.

**Returns:** `bool` — Whether the path is a directory.

In [7]:
print(files.is_dir(root / 'images'))
print(files.is_dir(root / 'images' / 'sample.jpg'))

True
False


**Another useful pattern**

In [8]:
for item in (root, root / 'images', root / 'images' / 'sample.jpg'):
    print(item.name, files.is_dir(item))

klygo-files-qossyejr True
images True
sample.jpg False


<a id="is_symlink"></a>

### `files.is_symlink`

Return whether a path is a symbolic link, including a broken link.

**Signature:** `files.is_symlink(path: 'PathInput') -> 'bool'`

- `path`: Path to inspect.

**Returns:** `bool` — True when the directory entry itself is a symbolic link.

In [9]:
print(files.is_symlink(root / 'images' / 'sample.jpg'))
print(files.is_symlink(root / 'images'))

False
False


**Another useful pattern**

In [10]:
link = root / 'image-link.jpg'
try:
    link.symlink_to(root / 'images' / 'sample.jpg')
except (OSError, NotImplementedError) as error:
    print('Symlinks unavailable here:', error)
else:
    print(files.is_symlink(link), files.is_file(link))

Symlinks unavailable here: [WinError 1314] A required privilege is not held by the client: 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-files-qossyejr\\images\\sample.jpg' -> 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-files-qossyejr\\image-link.jpg'


<a id="list_entries"></a>

### `files.list_entries`

List matching files and directories below a directory.

**Signature:** `files.list_entries(path: 'PathInput' = '.', pattern: 'str' = '*', recursive: 'bool' = False) -> 'List[Path]'`

- `path`: Path to inspect or operate on.
- `pattern`: Glob expression used to match entries.
- `recursive`: Include descendants.

**Returns:** `list[pathlib.Path]` — Matching files and directories in deterministic order.

This returns a sorted list of both files and directories. The glob pattern applies to entries, and `recursive=False` checks only immediate children.

In [11]:
print([p.relative_to(root).as_posix() for p in files.list_entries(root)])
print([p.relative_to(root).as_posix() for p in files.list_entries(root, '*.jpg', recursive=True)])

['cache', 'images', 'output']
['cache/ignored.jpg', 'images/sample.jpg', 'images/second.jpg']


**Another useful pattern**

In [12]:
for item in files.list_entries(root, recursive=False):
    print(item.name, 'directory' if files.is_dir(item) else 'file')

cache directory
images directory
output directory


**Expected error and recovery**

In [13]:
try:
    files.list_entries(root / "missing")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: Path does not exist: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\missing


<a id="find"></a>

### `files.find`

Find files matching a glob pattern below a directory.

**Signature:** `files.find(path: 'PathInput' = '.', pattern: 'str' = '*', recursive: 'bool' = True) -> 'List[Path]'`

- `path`: Path to inspect or operate on.
- `pattern`: Glob expression used to match entries.
- `recursive`: Include descendants.

**Returns:** `list[pathlib.Path]` — Matching regular files in deterministic order.

This returns a sorted list of regular files only. It scans recursively by default; use `recursive=False` for immediate children.

In [14]:
print([p.relative_to(root).as_posix() for p in files.find(root, '*.jpg')])

['cache/ignored.jpg', 'images/sample.jpg', 'images/second.jpg']


**Another useful pattern**

In [15]:
print('Only immediate images:', [p.name for p in files.find(root, '*.jpg', recursive=False)])
print('All nested images:', [p.name for p in files.find(root, '*.jpg')])

Only immediate images: []
All nested images: ['ignored.jpg', 'sample.jpg', 'second.jpg']


**Expected error and recovery**

In [16]:
try:
    files.find(root / "missing", "*.jpg")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: Path does not exist: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\missing


<a id="iter_find"></a>

### `files.iter_find`

Yield matching regular files without collecting the whole directory tree.

**Signature:** `files.iter_find(path: 'PathInput' = '.', pattern: 'str' = '*', recursive: 'bool' = True, *, exclude_dirs: 'Iterable[str]' = ()) -> 'Generator[Path, None, None]'`

- `path`: Existing directory at which to start searching.
- `pattern`: Filename wildcard (``*``, ``?``, ``[seq]``); no path separators.
- `recursive`: Include descendants.
- `exclude_dirs`: Directory basename wildcards to omit, such as ``('.git', 'cache*')``.

**Returns:** `generator[pathlib.Path, None, None]` — Matching files in filesystem order. Results are not sorted or reserved.

This yields files lazily in traversal order and can prune directory basenames via `exclude_dirs`. Its pattern matches a filename, not a path with separators; unlike `find()`, results are not sorted.

In [17]:
for image in files.iter_find(root, '*.jpg', exclude_dirs=('cache',)):
    print(image.relative_to(root).as_posix())

images/sample.jpg
images/second.jpg


**Another useful pattern**

In [18]:
generator = files.iter_find(root, '*.jpg', exclude_dirs=('cache',))
print('Iterator type:', type(generator).__name__)
print('Found:', [p.name for p in generator])

Iterator type: generator
Found: ['sample.jpg', 'second.jpg']


**Expected error and recovery**

In [19]:
try:
    files.iter_find(root, "images/*.jpg")
except ValueError as error:
    print("ValueError:", error)

print(list(files.iter_find(root, "*.jpg", recursive=False)))

ValueError: Patterns must be non-empty basenames without path separators
[]


<a id="walk"></a>

### `files.walk`

Walk a directory tree lazily.

**Signature:** `files.walk(path: 'PathInput' = '.') -> 'Generator[Tuple[str, List[str], List[str]], None, None]'`

- `path`: Path to inspect or operate on.

**Returns:** `iterator` — Tuples of root path, directory names, and file names.

This is a lazy `os.walk` iterator. It yields `(root_string, directory_names, file_names)`; it does not promise sorted names and does not validate a missing root before iteration.

In [20]:
for current, directories, names in files.walk(root):
    print(files.path(current).relative_to(root), sorted(directories), sorted(names))

. ['cache', 'images', 'output'] []
cache [] ['ignored.jpg']
images [] ['sample.jpg', 'second.jpg']
output [] ['result.json']


**Another useful pattern**

In [21]:
for current, directories, names in files.walk(root):
    if files.path(current) == root:
        print('Top-level directories:', sorted(directories))
        print('Top-level files:', sorted(names))

Top-level directories: ['cache', 'images', 'output']
Top-level files: []


<a id="mkdir"></a>

### `files.mkdir`

Create a directory and return its path.

**Signature:** `files.mkdir(path: 'PathInput', parents: 'bool' = True, exist_ok: 'bool' = True) -> 'Path'`

- `path`: Path to inspect or operate on.
- `parents`: Create missing parent directories.
- `exist_ok`: Do not fail when the directory already exists.

**Returns:** `pathlib.Path` — The created directory.

In [22]:
output_dir = files.mkdir(root / 'output' / 'labels')
print(output_dir, files.is_dir(output_dir))

C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\output\labels True


**Another useful pattern**

In [23]:
nested = files.mkdir(root / 'reports' / '2026')
print(nested, files.is_dir(nested))
print('Safe to call again:', files.mkdir(nested))

C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\reports\2026 True
Safe to call again: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\reports\2026


**Expected error and recovery**

In [24]:
try:
    files.mkdir(root / "images", exist_ok=False)
except FileExistsError as error:
    print("FileExistsError:", error)

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-files-qossyejr\\images'


<a id="copy"></a>

### `files.copy`

Copy a file or directory.

**Signature:** `files.copy(source: 'PathInput', target: 'PathInput', overwrite: 'bool' = False) -> 'Path'`

- `source`: Source URL or filesystem path.
- `target`: Destination filesystem path.
- `overwrite`: Allow an existing destination to be replaced.

**Returns:** `pathlib.Path` — Destination of the copied entry.

The source remains. The destination's parent is created if necessary; an existing entry raises `FileExistsError` unless `overwrite=True`. Do not copy a directory into itself.

In [25]:
source = root / 'images' / 'sample.jpg'
backup = files.copy(source, root / 'backup' / source.name)
print(backup, files.exists(source))
try:
    files.copy(source, backup)
except FileExistsError:
    print('Destination exists; choose another name or pass overwrite=True.')

C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\backup\sample.jpg True
Destination exists; choose another name or pass overwrite=True.


**Another useful pattern**

In [26]:
copied_tree = files.copy(root / 'images', root / 'images-copy')
print('Copied images:', [p.name for p in files.find(copied_tree, '*.jpg')])
print('Originals remain:', files.is_dir(root / 'images'))

Copied images: ['sample.jpg', 'second.jpg']
Originals remain: True


**Expected error and recovery**

In [27]:
destination = root / "backup" / "sample.jpg"
try:
    files.copy(root / "images" / "sample.jpg", destination)
except FileExistsError as error:
    print("FileExistsError:", error)

files.copy(root / "images" / "sample.jpg", destination, overwrite=True)
print("Replaced intentionally:", destination)

FileExistsError: Target already exists: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\backup\sample.jpg
Replaced intentionally: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\backup\sample.jpg


<a id="move"></a>

### `files.move`

Move a file or directory.

**Signature:** `files.move(source: 'PathInput', target: 'PathInput', overwrite: 'bool' = False) -> 'Path'`

- `source`: Source URL or filesystem path.
- `target`: Destination filesystem path.
- `overwrite`: Allow an existing destination to be replaced.

**Returns:** `pathlib.Path` — Destination of the moved entry.

The source is removed after the destination has been prepared. It can copy across filesystems; if removal then fails, both copies can remain.

In [28]:
draft = root / 'draft.txt'
draft.write_text('draft', encoding='utf-8')
final = files.move(draft, root / 'output' / 'final.txt')
print(final, files.exists(draft))

C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\output\final.txt False


**Another useful pattern**

In [29]:
draft = root / 'draft-2.txt'
draft.write_text('new result', encoding='utf-8')
destination = files.move(draft, root / 'reports' / 'result.txt')
print('Moved to:', destination, 'source exists:', files.exists(draft))

Moved to: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\reports\result.txt source exists: False


**Expected error and recovery**

In [30]:
try:
    files.move(root / "missing.txt", root / "output" / "missing.txt")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: Source does not exist: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\missing.txt


<a id="rename"></a>

### `files.rename`

Rename a filesystem entry or move it to an explicit path.

**Signature:** `files.rename(path: 'PathInput', new_name_or_path: 'PathInput', overwrite: 'bool' = False) -> 'Path'`

- `path`: Path to inspect or operate on.
- `new_name_or_path`: New basename or explicit destination path.
- `overwrite`: Allow an existing destination to be replaced.

**Returns:** `pathlib.Path` — New path of the renamed entry.

A single basename is resolved in the source directory; a multi-part path is treated as an explicit destination. Existing destinations require `overwrite=True`.

In [31]:
original = root / 'output' / 'final.txt'
renamed = files.rename(original, 'report.txt')
print(renamed, files.exists(original))

C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\output\report.txt False


**Another useful pattern**

In [32]:
old = root / 'rename-me.txt'
old.write_text('ready', encoding='utf-8')
new = files.rename(old, 'renamed.txt')
print('New basename:', new.name, 'old exists:', files.exists(old))

New basename: renamed.txt old exists: False


**Expected error and recovery**

In [33]:
current = root / "renamed.txt"
try:
    files.rename(current, current)
except ValueError as error:
    print("ValueError:", error)

ValueError: Source and destination must be different paths


<a id="remove"></a>

### `files.remove`

Remove a file, symbolic link, or directory.

**Signature:** `files.remove(path: 'PathInput', recursive: 'bool' = True, missing_ok: 'bool' = True) -> 'None'`

- `path`: Path to inspect or operate on.
- `recursive`: Remove non-empty directories recursively.
- `missing_ok`: Do not fail when the target is absent.

**Returns:** `None` — The target is removed as a side effect.

This is destructive and recursive by default. Use `recursive=False` when you intend to remove only an empty directory; a symlink itself is unlinked rather than its target.

In [34]:
disposable = files.mkdir(root / 'scratch' / 'nested')
(disposable / 'note.txt').write_text('temporary', encoding='utf-8')
files.remove(root / 'scratch')
print(files.exists(root / 'scratch'))
files.remove(root / 'already-missing.txt', missing_ok=True)

False


**Another useful pattern**

In [35]:
temporary_file = root / 'discard-me.txt'
temporary_file.write_text('temporary', encoding='utf-8')
files.remove(temporary_file, missing_ok=False)
print('Removed:', not files.exists(temporary_file))

Removed: True


**Expected error and recovery**

In [36]:
try:
    files.remove(root / "missing.txt", missing_ok=False)
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: Path does not exist: C:\Users\Mine\AppData\Local\Temp\klygo-files-qossyejr\missing.txt


## Put it together

In [37]:
files.mkdir(root / 'selected')
for image in files.iter_find(root / 'images', '*.jpg'):
    files.copy(image, root / 'selected' / image.name)
print([p.name for p in files.find(root / 'selected', '*.jpg')])

['sample.jpg', 'second.jpg']


## Remember

- Use `find()` for a sorted list and `iter_find()` for a lazy scan; check destinations before passing `overwrite=True` to mutation functions.
- Next in `klygo.files`: continue with [Document_Files_Metadata.ipynb](Document_Files_Metadata.ipynb) ([Colab](https://colab.research.google.com/drive/1DOXIerhqeNEgUKp712HAwSxI4X8uwyOW?usp=sharing)) or return to [Document_Files_Overview.ipynb](Document_Files_Overview.ipynb) ([Colab](https://colab.research.google.com/drive/12PhuA8SVe_Q8jfC4f5klsgQifR6Vdf4H?usp=sharing)).
- Related Klygo modules: [`klygo.archive`](../archive/Document_Archive_Overview.ipynb) · [`klygo.config`](../config/Document_Config_Overview.ipynb).